In [31]:
import pandas as pd
import joblib
import time
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [32]:
project_root = Path.cwd()

if not (project_root / "data" / "data.csv").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "data.csv"

df = pd.read_csv(data_path)

print("Jumlah data awal:", len(df))
print("\nJumlah data kosong:")
print(df.isnull().sum())
print("\nDistribusi label:")
print(df["Label"].value_counts())

Jumlah data awal: 4009

Jumlah data kosong:
URLs         0
Headline     0
Body        21
Label        0
dtype: int64

Distribusi label:
Label
0    2137
1    1872
Name: count, dtype: int64


In [33]:
df = df.dropna(subset=["Headline", "Label"]).copy()

df["Headline"] = (
    df["Headline"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[df["Headline"] != ""].copy()

df["headline_key"] = df["Headline"].str.casefold()

konflik = df.groupby("headline_key")["Label"].nunique()

if (konflik > 1).any():
    raise ValueError("Ada judul yang memiliki label berbeda.")

df = df.drop_duplicates(
    subset=["headline_key"]
).reset_index(drop=True)

print("Jumlah data setelah preprocessing:", len(df))
print("\nDistribusi label:")
print(df["Label"].value_counts())

Jumlah data setelah preprocessing: 2831

Distribusi label:
Label
1    1605
0    1226
Name: count, dtype: int64


In [34]:
X = df["Headline"]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah seluruh data:", len(X))
print("Jumlah data latih:", len(X_train))
print("Jumlah data uji:", len(X_test))

Jumlah seluruh data: 2831
Jumlah data latih: 2264
Jumlah data uji: 567


In [35]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=10000
    )),
    ("svm", LinearSVC(
        C=1.0,
        random_state=42
    ))
])

mulai = time.perf_counter()

model.fit(X_train, y_train)

waktu_training = time.perf_counter() - mulai

print("Training model berhasil!")
print(f"Waktu pelatihan: {waktu_training:.4f} detik")

Training model berhasil!
Waktu pelatihan: 0.0737 detik


In [36]:
mulai_prediksi = time.perf_counter()

y_pred = model.predict(X_test)

waktu_prediksi = time.perf_counter() - mulai_prediksi

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test, y_pred, average="weighted", zero_division=0
)
recall = recall_score(
    y_test, y_pred, average="weighted", zero_division=0
)
f1 = f1_score(
    y_test, y_pred, average="weighted", zero_division=0
)

print("HASIL EVALUASI MODEL NEWSCHECK")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"Waktu prediksi: {waktu_prediksi:.4f} detik")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["FAKE", "REAL"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

HASIL EVALUASI MODEL NEWSCHECK
Accuracy  : 0.7760
Precision : 0.7752
Recall    : 0.7760
F1-Score  : 0.7747
Waktu prediksi: 0.0176 detik

Classification Report:
              precision    recall  f1-score   support

        FAKE       0.76      0.70      0.73       246
        REAL       0.79      0.83      0.81       321

    accuracy                           0.78       567
   macro avg       0.77      0.77      0.77       567
weighted avg       0.78      0.78      0.77       567


Confusion Matrix:
[[173  73]
 [ 54 267]]


In [37]:
model_path = project_root / "model" / "newscheck_pipeline.joblib"

model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model, model_path)

print("Model berhasil disimpan!")
print("Lokasi:", model_path)

Model berhasil disimpan!
Lokasi: c:\Users\Luthfi\Downloads\NewsCheck-UAS-ML\model\newscheck_pipeline.joblib


In [38]:
model_tersimpan = joblib.load(model_path)

prediksi_awal = model.predict(X_test.iloc[:5])
prediksi_simpan = model_tersimpan.predict(X_test.iloc[:5])

assert (prediksi_awal == prediksi_simpan).all()

print("Model berhasil dimuat kembali!")
print("Prediksi model:", prediksi_simpan)
print("Pengujian penyimpanan model berhasil!")

Model berhasil dimuat kembali!
Prediksi model: [0 1 1 0 0]
Pengujian penyimpanan model berhasil!
